# Notebook 05 — Computational Analysis (Simulation)

**Dataset:** `issues_clean.csv`  
**Variabel:** `age_days`  
**Research Question (RQ3):** Berapakah probabilitas sebuah issue di moby/moby membutuhkan lebih dari 60 hari untuk ditutup?  
**Member:** Audylia Aska Widyaputri — Computation Analyst (Member E)  
**NIM:** 1519625036

## AI Usage Disclosure

**Member:** Audylia Aska Widyaputri — Computation Analyst | **Tools used:** Claude (Anthropic)

| Task | Tool | Prompt summary | Output |
|------|------|----------------|-----------------|
| Diskusi pendekatan simulasi | Claude | "Metode Monte Carlo yang cocok untuk dataset issue tracking?" | Ya, disesuaikan dengan dataset |
| Validasi logika program | Claude | "Apakah logika perhitungan probabilitas sudah benar?" | Ya, diterapkan setelah pengecekan manual |

**Ditulis sepenuhnya tanpa AI:** implementasi kode, analisis hasil simulasi, interpretasi output, dan kesimpulan.

## Monte Carlo Simulation

### Langkah-langkah simulasi
1. Import library yang dibutuhkan
2. Load dan filter data closed issues
3. Jalankan simulasi bootstrap (50.000 iterasi)
4. Hitung estimasi probabilitas dan 95% CI
5. Visualisasi hasil

### 1. Import library yang dibutuhkan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import random
import math

### 2. Load dan filter data closed issues

Hanya gunakan issue dengan `state == 'closed'` dan memiliki `closed_at` yang valid. Kolom `age_days` sudah berisi durasi issue dalam hari.

In [ ]:
df = pd.read_csv('../data/clean/issues_clean.csv')
df_closed = df[(df['state'] == 'closed') & (df['closed_at'].notna())]
age_days = df_closed['age_days'].dropna().astype(float).values
print(f'Jumlah closed issues: {len(age_days)}')

### 3. Jalankan simulasi Monte Carlo (50.000 iterasi)

Setiap iterasi mengambil resample seluruh data dengan pengembalian (`replace=True`), lalu hitung proporsi yang > 60 hari.

In [ ]:
n_sim = 50000
sample_size = len(age_days)
results = np.empty(n_sim)
for i in range(n_sim):
    sample = np.random.choice(age_days, size=sample_size, replace=True)
    results[i] = np.mean(sample > 60)

### 4. Hitung estimasi probabilitas dan 95% Confidence Interval

In [ ]:
estimate = results.mean()
ci_lower, ci_upper = np.percentile(results, [2.5, 97.5])
print(f'Estimated Probability: {estimate:.4f}')
print(f'95% Confidence Interval: {ci_lower:.4f} - {ci_upper:.4f}')

### 5. Visualisasi hasil simulasi

In [ ]:
plt.hist(results, bins=30, edgecolor='black')
plt.axvline(estimate, color='red', linestyle='--', label=f'estimasi = {estimate:.4f}')
plt.title('Monte Carlo Simulation Results')
plt.xlabel('Probability Issue > 60 Days')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

## Bloom Filter

Bloom Filter digunakan untuk mengecek apakah suatu issue URL pernah diproses sebelumnya secara efisien.

### Langkah-langkah
1. Definisikan class BloomFilter dengan k fungsi hash dan m ukuran bit array
2. Masukkan seluruh URL issue ke Bloom Filter
3. Uji item yang ada dan yang tidak ada
4. Hitung False Positive Rate (FPR) teoritis

### 1. Definisikan class BloomFilter

In [ ]:
class BloomFilter:
    def __init__(self, k, m):
        self.k = k
        self.m = m
        self.bit_array = [0] * m
        self.n_items = 0

    def _hash(self, item, seed):
        key = f"{seed}:{item}"
        return int(hashlib.sha256(key.encode()).hexdigest(), 16) % self.m

    def add(self, item):
        for seed in range(self.k):
            self.bit_array[self._hash(item, seed)] = 1
        self.n_items += 1

    def contains(self, item):
        return all(self.bit_array[self._hash(item, seed)] == 1 for seed in range(self.k))

    def theoretical_fpr(self, n):
        return round((1 - (1 - 1/self.m)**n)**self.k, 8)

### 2. Masukkan seluruh URL issue ke Bloom Filter

In [ ]:
bf = BloomFilter(k=5, m=5000)
for url in df['url'].tolist():
    bf.add(str(url))
print(f'Total issue dimasukkan: {bf.n_items}')

### 3. Uji item yang ada dan yang tidak ada dalam dataset

In [ ]:
# item yang tidak ada (seharusnya False)
test_fake = ['https://github.com/moby/moby/issues/99999',
             'https://github.com/moby/moby/issues/00001']
for url in test_fake:
    print(f'{url} → {bf.contains(url)}')

### 4. Hitung False Positive Rate teoritis

Formula: FPR = (1 - (1 - 1/m)^n)^k  (Tsun, 2020, p.329)

In [ ]:
fpr = bf.theoretical_fpr(bf.n_items)
print(f'FPR teoritis (k=5, m=5000, n={bf.n_items}): {fpr:.6f} ({fpr*100:.4f}%)')

## MCMC Knapsack

MCMC (Metropolis-Hastings) digunakan untuk mencari subset issue optimal dalam kapasitas review terbatas.
- **Bobot (weight):** `age_days` — durasi penyelesaian issue
- **Nilai (value):** `comments` — proxy tingkat kepentingan komunitas
- **Kapasitas:** 120 hari

### Langkah-langkah
1. Siapkan item dari closed issues
2. Jalankan MCMC Metropolis-Hastings (100.000 iterasi)
3. Tampilkan hasil issue terpilih

### 1. Siapkan item dari closed issues

In [ ]:
df_mcmc = df_closed[df_closed['age_days'] > 0].copy()
df_mcmc['comments'] = pd.to_numeric(df_mcmc['comments'], errors='coerce').fillna(0)

items = [{'name': f"issue_{r['number']}", 'weight': float(r['age_days']), 'value': float(r['comments'])}
         for _, r in df_mcmc.iterrows()]

print(f'Jumlah item: {len(items)} | Kapasitas: 120 hari')

### 2. Jalankan MCMC Metropolis-Hastings (100.000 iterasi)

In [ ]:
capacity = 120
n_iter = 100000
random.seed(42)
n = len(items)
current = [0] * n
best_state = current[:]
best_value = 0
T = 1.0
accepted = 0

for _ in range(n_iter):
    proposal = current[:]
    proposal[random.randint(0, n-1)] ^= 1
    w = sum(items[i]['weight'] for i in range(n) if proposal[i])
    if w > capacity:
        continue
    v_cur = sum(items[i]['value'] for i in range(n) if current[i])
    v_pro = sum(items[i]['value'] for i in range(n) if proposal[i])
    if v_pro >= v_cur or random.random() < math.exp((v_pro - v_cur) / T):
        current = proposal
        accepted += 1
    if sum(items[i]['value'] for i in range(n) if current[i]) > best_value:
        best_state = current[:]
        best_value = sum(items[i]['value'] for i in range(n) if best_state[i])
    T = max(0.01, T * 0.99995)

### 3. Tampilkan hasil issue terpilih

In [ ]:
selected = [items[i]['name'] for i in range(n) if best_state[i] == 1]
total_w = sum(items[i]['weight'] for i in range(n) if best_state[i])
print(f'Issue terpilih : {len(selected)}')
print(f'Total nilai    : {best_value:.0f} komentar')
print(f'Total bobot    : {total_w:.0f} / {capacity} hari')
print(f'Acceptance rate: {accepted/n_iter:.4f}')